In [1]:
import requests
import zipfile
import io
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import subprocess
import time
from tqdm import tqdm

# Configurations

In [2]:
# Download URL for latest NHS GP data
download_url = "https://digital.nhs.uk/binaries/content/assets/website-assets/services/organisation-data-service/data-downloads/gp-and-gp-practice-related-data/epraccur.zip"

# Custom header list from metadata (27 columns)
headers = [
    "organisation_code", "name", "national_grouping", "high_level_health_geography",
    "address_line_1", "address_line_2", "address_line_3", "address_line_4", "address_line_5", "postcode",
    "open_date", "close_date", "status_code", "org_sub_type_code", "commissioner",
    "join_provider_date", "left_provider_date", "telephone", "null_1", "null_2", "null_3",
    "amended_record", "null_4", "provider_code", "null_5", "prescribing_setting", "null_6"
]

# Path and database config
layer_name = "nhs_gp_practices"
ogr2ogr_path = r"C:\Program Files\QGIS 3.34.11\bin\ogr2ogr.exe"
gpkg_path = r"N:\Geodatabase\Raw_Data\GPKG_temp_folder_for_geodatabse_upload/temp_upload.gpkg"
target_crs = "EPSG:27700"
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
pg_conn_string = f"PG:host={db_host} dbname={db_name} port={db_port}"

# Download and extract the CSV

In [ ]:
response = requests.get(download_url)
if response.status_code != 200:
    raise Exception("Failed to download epraccur ZIP")

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files in archive:", z.namelist())
    csv_file = [f for f in z.namelist() if f.lower().endswith(".csv")][0]
    
    with z.open(csv_file) as f:
        df = pd.read_csv(f, header=None, names=headers, dtype=str)

print(f"Total rows loaded: {len(df)}")

# Filter, clean columns, and convert to GeoDataFrame

In [ ]:
# Filter only open establishments
df = df[df["EstablishmentStatus (name)"].str.lower() == "open"].copy()
print(f"Remaining rows after filtering 'open': {len(df)}")

# Rename columns: replace spaces with underscores and lowercase
df.columns = [c.strip().lower().replace(" ", "_").replace("(", "").replace(")", "") for c in df.columns]

# Build geometry from easting/northing and convert to GeoDataFrame

In [21]:
# Drop missing coordinate rows
df = df.dropna(subset=["easting", "northing"])

# Convert to geometry
geometry = [Point(xy) for xy in zip(df["easting"], df["northing"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=target_crs)

# Save to GeoPackage
gdf.to_file(gpkg_path, driver="GPKG", layer=layer_name)
print("GeoPackage written:", gpkg_path)


# Upload to PostGIS using ogr2ogr

In [22]:
print("Uploading to PostGIS...")
start_time = time.time()

ogr2ogr_cmd = [
    ogr2ogr_path,
    "-f", "PostgreSQL",
    pg_conn_string,
    gpkg_path,
    "-nln", f"{db_schema}.{layer_name}",
    "-nlt", "PROMOTE_TO_MULTI",
    "-lco", "GEOMETRY_NAME=geometry",
    "-lco", "SPATIAL_INDEX=GIST",
    "-lco", "SCHEMA=" + db_schema,
    "-lco", "OVERWRITE=YES",
    "-overwrite"
]

with tqdm(total=100, desc=f"Uploading {layer_name}", bar_format="{l_bar}{bar}| {elapsed}") as pbar:
    result = subprocess.run(ogr2ogr_cmd, capture_output=True, text=True)
    pbar.update(100)

if result.returncode != 0:
    print("❌ Upload failed:")
    print(result.stderr)
else:
    print("✅ Upload successful.")
    print(f"⏱ Time taken: {time.time() - start_time:.2f} seconds")


Reading input layer...
Detected input CRS: 4326
Target CRS: 27700
Reprojecting to target CRS...
Converting .geojson to GeoPackage for upload...


C:\Users\abhimanya.achara\Anaconda3\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


Uploading to PostGIS via ogr2ogr...


Uploading: 100%|██████████| 24:24


 Upload completed successfully.
 Time taken: 1464.76 seconds
